# Generate Raw Data for `ronguerrero.cpp_investment_ops`

This notebook creates synthetic investment operations data to feed the CPP_SDP pipeline.
Intentionally includes bad records (nulls, negatives, invalid ranges) to exercise data quality expectations.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ronguerrero.cpp_investment_ops

In [0]:
import random
import uuid
from datetime import datetime, date, timedelta
from pyspark.sql import Row
from pyspark.sql.types import *
import pyspark.sql.functions as F

random.seed(42)
SCHEMA = "ronguerrero.cpp_investment_ops"

In [0]:
# --- raw_securities_master: 24 securities with 2 intentionally bad records ---

securities = [
    ("SEC-001", "US0378331005", "AAPL", "Apple Inc", "Equity", "Technology", "US", "USD", "NASDAQ", "Apple Inc", None, None, 82.5),
    ("SEC-002", "US5949181045", "MSFT", "Microsoft Corp", "Equity", "Technology", "US", "USD", "NASDAQ", "Microsoft Corp", None, None, 88.1),
    ("SEC-003", "US0231351067", "AMZN", "Amazon.com Inc", "Equity", "Consumer Discretionary", "US", "USD", "NASDAQ", "Amazon.com Inc", None, None, 65.3),
    ("SEC-004", "US30303M1027", "META", "Meta Platforms Inc", "Equity", "Technology", "US", "USD", "NASDAQ", "Meta Platforms", None, None, 71.0),
    ("SEC-005", "US88160R1014", "TSLA", "Tesla Inc", "Equity", "Consumer Discretionary", "US", "USD", "NASDAQ", "Tesla Inc", None, None, 55.8),
    ("SEC-006", "GB0002374006", "DGE.L", "Diageo PLC", "Equity", "Consumer Staples", "GB", "GBP", "LSE", "Diageo PLC", None, None, 79.2),
    ("SEC-007", "DE0007164600", "SAP.DE", "SAP SE", "Equity", "Technology", "DE", "EUR", "XETRA", "SAP SE", None, None, 84.6),
    ("SEC-008", "JP3633400001", "7203.T", "Toyota Motor Corp", "Equity", "Industrials", "JP", "JPY", "TSE", "Toyota Motor", None, None, 72.1),
    ("SEC-009", "FR0000120271", "TTE.PA", "TotalEnergies SE", "Equity", "Energy", "FR", "EUR", "Euronext", "TotalEnergies", None, None, 45.3),
    ("SEC-010", "CH0012032048", "ROG.SW", "Roche Holding AG", "Equity", "Healthcare", "CH", "CHF", "SIX", "Roche Holding", None, None, 90.4),
    ("SEC-011", "US912828ZT09", "T-NOTE-5Y", "US Treasury 5Y Note", "Fixed Income", "Government", "US", "USD", "OTC", "US Treasury", date(2029, 6, 15), 3.75, None),
    ("SEC-012", "US912810SZ21", "T-BOND-30Y", "US Treasury 30Y Bond", "Fixed Income", "Government", "US", "USD", "OTC", "US Treasury", date(2054, 11, 15), 4.25, None),
    ("SEC-013", "XS2345678901", "GS-CORP-5Y", "Goldman Sachs 5Y Corp", "Fixed Income", "Financials", "US", "USD", "OTC", "Goldman Sachs", date(2029, 3, 1), 5.10, 68.0),
    ("SEC-014", "DE0001102481", "DBR-10Y", "German Bund 10Y", "Fixed Income", "Government", "DE", "EUR", "XETRA", "German Govt", date(2034, 8, 15), 2.50, None),
    ("SEC-015", "GB00BN65R198", "UKT-5Y", "UK Gilt 5Y", "Fixed Income", "Government", "GB", "GBP", "LSE", "UK DMO", date(2029, 9, 7), 4.00, None),
    ("SEC-016", "US46625HJK86", "JPM-FRN", "JPMorgan Chase FRN", "Fixed Income", "Financials", "US", "USD", "OTC", "JPMorgan Chase", date(2027, 12, 1), 5.85, 73.5),
    ("SEC-017", "IE00B4L5Y983", "IWDA.AS", "iShares MSCI World ETF", "ETF", "Diversified", "IE", "USD", "Euronext", "BlackRock", None, None, 85.0),
    ("SEC-018", "US78462F1030", "SPY", "SPDR S&P 500 ETF", "ETF", "Diversified", "US", "USD", "NYSE Arca", "State Street", None, None, 80.2),
    ("SEC-019", "IE00B1XNHC34", "AGGG.L", "iShares Global Agg Bond", "ETF", "Fixed Income", "IE", "USD", "LSE", "BlackRock", None, None, 76.8),
    ("SEC-020", "US4642874659", "IVV", "iShares Core S&P 500", "ETF", "Diversified", "US", "USD", "NYSE Arca", "BlackRock", None, None, 81.5),
    ("SEC-021", "FXFWD-USDEUR", None, "USD/EUR FX Forward 3M", "FX Forward", "Currency", "US", "USD", "OTC", "Citi", date(2026, 8, 14), None, None),
    ("SEC-022", "FXFWD-GBPUSD", None, "GBP/USD FX Forward 6M", "FX Forward", "Currency", "GB", "GBP", "OTC", "Barclays", date(2026, 11, 14), None, None),
    # --- BAD RECORDS ---
    (None, "US0000000000", "BAD1", "Bad Security 1", "Equity", "Technology", "US", "USD", "NYSE", "Bad Issuer", None, None, 75.0),
    ("SEC-BAD-02", None, None, None, None, "Finance", "UK", "GBP", "LSE", "Another Issuer", date(2028, 6, 15), 0.03, 60.0),
]

schema_sec = StructType([
    StructField("security_id", StringType()),
    StructField("isin", StringType()),
    StructField("ticker", StringType()),
    StructField("security_name", StringType()),
    StructField("asset_class", StringType()),
    StructField("sector", StringType()),
    StructField("country", StringType()),
    StructField("currency", StringType()),
    StructField("exchange", StringType()),
    StructField("issuer_name", StringType()),
    StructField("maturity_date", DateType()),
    StructField("coupon_rate", DoubleType()),
    StructField("esg_score", DoubleType()),
])

df_sec = spark.createDataFrame(securities, schema=schema_sec)
df_sec.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_securities_master")
print(f"raw_securities_master: {df_sec.count()} rows")

In [0]:
# --- raw_counterparties: 8 counterparties with 2 bad records ---

counterparties = [
    ("CP-001", "Goldman Sachs International", "Broker", "W22LROWP2IHZNBB6K528", "US", "A+"),
    ("CP-002", "JPMorgan Securities LLC", "Broker", "8I5DZWZKVSZI1NUHU748", "US", "AA-"),
    ("CP-003", "Barclays Capital Securities", "Broker", "AC28XWWI3WIBK2824F44", "GB", "A"),
    ("CP-004", "BNP Paribas Securities", "Broker", "R0MUWSFPU8MPRO8K5P83", "FR", "A+"),
    ("CP-005", "State Street Bank & Trust", "Custodian", "571474TGEMMWANRLN572", "US", "AA-"),
    ("CP-006", "Northern Trust Company", "Custodian", "6PTKHDJ8HDUF78PFWH30", "US", "A+"),
    # --- BAD RECORDS ---
    (None, "Ghost Bank", "Broker", "LEI000000000000000000", "US", "AA"),
    ("CP-BAD-02", None, "Custodian", None, "UK", None),
]

schema_cp = StructType([
    StructField("counterparty_id", StringType()),
    StructField("counterparty_name", StringType()),
    StructField("counterparty_type", StringType()),
    StructField("lei", StringType()),
    StructField("country", StringType()),
    StructField("credit_rating", StringType()),
])

df_cp = spark.createDataFrame(counterparties, schema=schema_cp)
df_cp.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_counterparties")
print(f"raw_counterparties: {df_cp.count()} rows")

In [0]:
# --- raw_fund_structures: 5 funds with 2 bad records ---

funds = [
    ("FUND-001", "Global Alpha Growth Fund", "Growth", "BM-SPX", date(2018, 3, 15), 0.12, "USD"),
    ("FUND-002", "Fixed Income Total Return", "Income", "BM-AGG", date(2019, 7, 1), 0.06, "USD"),
    ("FUND-003", "European Value Equity", "Value", "BM-STOXX600", date(2020, 1, 10), 0.09, "EUR"),
    ("FUND-004", "Asia Pacific Opportunities", "Growth", "BM-MSCIAPAC", date(2021, 4, 20), 0.10, "USD"),
    ("FUND-005", "Multi-Asset Balanced Fund", "Balanced", "BM-6040", date(2017, 11, 1), 0.07, "GBP"),
    # --- BAD RECORDS ---
    (None, "Phantom Fund", "Growth", "BM-001", date(2025, 1, 1), 0.08, "USD"),
    ("FUND-BAD-02", None, "Value", "BM-002", None, -0.05, "EUR"),
]

schema_fund = StructType([
    StructField("fund_id", StringType()),
    StructField("fund_name", StringType()),
    StructField("strategy", StringType()),
    StructField("benchmark_id", StringType()),
    StructField("inception_date", DateType()),
    StructField("target_return", DoubleType()),
    StructField("base_currency", StringType()),
])

df_fund = spark.createDataFrame(funds, schema=schema_fund)
df_fund.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_fund_structures")
print(f"raw_fund_structures: {df_fund.count()} rows")

In [0]:
# --- raw_trade_executions: ~30,000 trades with ~900 bad records ---

valid_security_ids = [f"SEC-{str(i).zfill(3)}" for i in range(1, 23)]
valid_cp_ids = [f"CP-{str(i).zfill(3)}" for i in range(1, 7)]
valid_fund_ids = [f"FUND-{str(i).zfill(3)}" for i in range(1, 6)]
trade_types = ["BUY", "SELL"]
currencies = ["USD", "EUR", "GBP", "JPY", "CHF"]
brokers = ["BRK-01", "BRK-02", "BRK-03", "BRK-04", "BRK-05"]
venues = ["NYSE", "NASDAQ", "LSE", "XETRA", "Euronext", "TSE", "OTC", "Dark Pool"]
asset_classes = ["Equity", "Fixed Income", "ETF", "FX Forward"]
settlement_statuses = ["Settled", "Pending", "Failed", "Partially Settled"]

start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 5, 14)
date_range_seconds = int((end_date - start_date).total_seconds())

trades = []
for i in range(30000):
    # ~3% chance of a bad record
    is_bad = random.random() < 0.03
    
    trade_id = None if (is_bad and random.random() < 0.33) else f"TRD-{str(i+1).zfill(7)}"
    order_id = f"ORD-{str(i+1).zfill(7)}"
    security_id = random.choice(valid_security_ids)
    counterparty_id = random.choice(valid_cp_ids)
    fund_id = random.choice(valid_fund_ids)
    
    if is_bad and random.random() < 0.1:
        trade_date = None
    else:
        rand_seconds = random.randint(0, date_range_seconds)
        trade_date = start_date + timedelta(seconds=rand_seconds)
    
    trade_type = random.choice(trade_types)
    
    if is_bad and random.random() < 0.5:
        quantity = round(random.uniform(-1000, -1), 2)
    else:
        quantity = round(random.uniform(10, 50000), 2)
    
    if is_bad and random.random() < 0.5:
        price = round(random.uniform(-100, -0.01), 4)
    else:
        price = round(random.uniform(1, 5000), 4)
    
    currency = None if (is_bad and random.random() < 0.1) else random.choice(currencies)
    broker_id = random.choice(brokers)
    venue = random.choice(venues)
    asset_class = random.choice(asset_classes)
    settlement_status = random.choice(settlement_statuses)
    
    trades.append((
        trade_id, order_id, security_id, counterparty_id, fund_id,
        trade_date, trade_type, quantity, price, currency,
        broker_id, venue, asset_class, settlement_status
    ))

schema_trades = StructType([
    StructField("trade_id", StringType()),
    StructField("order_id", StringType()),
    StructField("security_id", StringType()),
    StructField("counterparty_id", StringType()),
    StructField("fund_id", StringType()),
    StructField("trade_date", TimestampType()),
    StructField("trade_type", StringType()),
    StructField("quantity", DoubleType()),
    StructField("price", DoubleType()),
    StructField("currency", StringType()),
    StructField("broker_id", StringType()),
    StructField("execution_venue", StringType()),
    StructField("asset_class", StringType()),
    StructField("settlement_status", StringType()),
])

df_trades = spark.createDataFrame(trades, schema=schema_trades)
df_trades.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_trade_executions")
print(f"raw_trade_executions: {df_trades.count()} rows")
print(f"  null trade_ids: {df_trades.filter(F.col('trade_id').isNull()).count()}")
print(f"  negative quantities: {df_trades.filter(F.col('quantity') < 0).count()}")
print(f"  negative prices: {df_trades.filter(F.col('price') < 0).count()}")

In [0]:
# --- raw_market_data: ~5,250 rows (250 days x 21 securities) with bad records ---

market_securities = [f"SEC-{str(i).zfill(3)}" for i in range(1, 22)]  # exclude bad securities
base_prices = {
    "SEC-001": 185.0, "SEC-002": 420.0, "SEC-003": 178.0, "SEC-004": 510.0,
    "SEC-005": 245.0, "SEC-006": 28.50, "SEC-007": 195.0, "SEC-008": 22.0,
    "SEC-009": 58.0, "SEC-010": 265.0, "SEC-011": 98.5, "SEC-012": 95.2,
    "SEC-013": 99.8, "SEC-014": 97.5, "SEC-015": 98.0, "SEC-016": 100.2,
    "SEC-017": 82.0, "SEC-018": 530.0, "SEC-019": 5.2, "SEC-020": 545.0,
    "SEC-021": 1.08, "SEC-022": 1.27,
}
currency_map = {
    "SEC-001": "USD", "SEC-002": "USD", "SEC-003": "USD", "SEC-004": "USD",
    "SEC-005": "USD", "SEC-006": "GBP", "SEC-007": "EUR", "SEC-008": "JPY",
    "SEC-009": "EUR", "SEC-010": "CHF", "SEC-011": "USD", "SEC-012": "USD",
    "SEC-013": "USD", "SEC-014": "EUR", "SEC-015": "GBP", "SEC-016": "USD",
    "SEC-017": "USD", "SEC-018": "USD", "SEC-019": "USD", "SEC-020": "USD",
    "SEC-021": "USD", "SEC-022": "GBP",
}
sources = ["Bloomberg", "Reuters", "ICE"]

market_data = []
start_md = date(2025, 7, 1)

for day_offset in range(250):
    current_date = start_md + timedelta(days=day_offset)
    # Skip weekends
    if current_date.weekday() >= 5:
        continue
    
    for sec_id in market_securities:
        is_bad = random.random() < 0.05  # 5% bad records
        
        base = base_prices[sec_id]
        daily_change = random.uniform(-0.03, 0.03)
        open_price = round(base * (1 + random.uniform(-0.02, 0.02)), 4)
        close_price = round(open_price * (1 + daily_change), 4)
        high_price = round(max(open_price, close_price) * (1 + random.uniform(0, 0.015)), 4)
        low_price = round(min(open_price, close_price) * (1 - random.uniform(0, 0.015)), 4)
        volume = random.randint(100000, 50000000)
        
        # Inject bad data
        sid = sec_id
        pd = current_date
        if is_bad:
            bad_type = random.choice(["null_id", "neg_price", "neg_volume", "inverted_range"])
            if bad_type == "null_id":
                sid = None
            elif bad_type == "neg_price":
                close_price = round(-abs(close_price), 4)
            elif bad_type == "neg_volume":
                volume = -abs(volume)
            elif bad_type == "inverted_range":
                low_price, high_price = high_price, low_price  # low > high
        
        market_data.append((
            sid, pd, close_price, open_price, high_price, low_price,
            volume, currency_map.get(sec_id, "USD"), random.choice(sources)
        ))

# Add a couple of explicit bad records
market_data.append((None, date(2026, 5, 14), 150.0, 148.0, 152.0, 147.0, 1000000, "USD", "Bloomberg"))
market_data.append(("SEC-BAD-MD2", None, -5.50, 100.0, 105.0, 95.0, -500, "EUR", "Reuters"))

schema_md = StructType([
    StructField("security_id", StringType()),
    StructField("price_date", DateType()),
    StructField("close_price", DoubleType()),
    StructField("open_price", DoubleType()),
    StructField("high_price", DoubleType()),
    StructField("low_price", DoubleType()),
    StructField("volume", LongType()),
    StructField("currency", StringType()),
    StructField("source", StringType()),
])

df_md = spark.createDataFrame(market_data, schema=schema_md)
df_md.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_market_data")
print(f"raw_market_data: {df_md.count()} rows")
print(f"  null security_ids: {df_md.filter(F.col('security_id').isNull()).count()}")
print(f"  negative close_prices: {df_md.filter(F.col('close_price') < 0).count()}")
print(f"  negative volumes: {df_md.filter(F.col('volume') < 0).count()}")

In [0]:
# --- raw_fx_rates: ~1,300 rows with bad records ---

fx_pairs = [
    ("USD", "EUR", 0.92), ("USD", "GBP", 0.79), ("USD", "JPY", 154.5),
    ("USD", "CHF", 0.88), ("EUR", "GBP", 0.86), ("EUR", "JPY", 168.0),
    ("GBP", "EUR", 1.16), ("GBP", "USD", 1.27),
]
fx_sources = ["ECB", "Reuters", "Bloomberg", "Fed"]

fx_rates = []
start_fx = date(2025, 1, 1)

for day_offset in range(500):
    current_date = start_fx + timedelta(days=day_offset)
    if current_date.weekday() >= 5:
        continue
    
    for from_ccy, to_ccy, base_rate in fx_pairs:
        is_bad = random.random() < 0.04  # 4% bad records
        
        daily_var = random.uniform(-0.005, 0.005)
        rate = round(base_rate * (1 + daily_var), 6)
        rd = current_date
        fc = from_ccy
        tc = to_ccy
        
        if is_bad:
            bad_type = random.choice(["null_date", "null_currency", "neg_rate", "zero_rate"])
            if bad_type == "null_date":
                rd = None
            elif bad_type == "null_currency":
                fc = None
                tc = None
            elif bad_type == "neg_rate":
                rate = round(-abs(rate), 6)
            elif bad_type == "zero_rate":
                rate = 0.0
        
        fx_rates.append((rd, fc, tc, rate, random.choice(fx_sources)))

schema_fx = StructType([
    StructField("rate_date", DateType()),
    StructField("from_currency", StringType()),
    StructField("to_currency", StringType()),
    StructField("rate", DoubleType()),
    StructField("source", StringType()),
])

df_fx = spark.createDataFrame(fx_rates, schema=schema_fx)
df_fx.write.mode("overwrite").saveAsTable(f"{SCHEMA}.raw_fx_rates")
print(f"raw_fx_rates: {df_fx.count()} rows")
print(f"  null rate_dates: {df_fx.filter(F.col('rate_date').isNull()).count()}")
print(f"  null currencies: {df_fx.filter(F.col('from_currency').isNull()).count()}")
print(f"  non-positive rates: {df_fx.filter(F.col('rate') <= 0).count()}")

In [0]:
# --- Summary: verify all tables ---

tables = [
    "raw_trade_executions", "raw_securities_master", "raw_market_data",
    "raw_fx_rates", "raw_counterparties", "raw_fund_structures"
]

print("=" * 60)
print(f"{'Table':<30} {'Rows':>10}")
print("=" * 60)
for t in tables:
    count = spark.table(f"{SCHEMA}.{t}").count()
    print(f"{t:<30} {count:>10,}")
print("=" * 60)
print("\nAll raw tables created successfully in ronguerrero.cpp_investment_ops")